# Modelagem Supervisionada

Dois problemas de aprendizado supervisionado:

1. **Classificação binária**: Prever se um voo será atrasado (`IS_DELAYED` = DEPARTURE_DELAY > 15 min)
   - Logistic Regression (baseline) vs XGBoost
2. **Regressão**: Prever a magnitude do atraso (`DEPARTURE_DELAY` em minutos, para voos com delay > 0)
   - Linear Regression (baseline) vs LightGBM vs XGBoost

**20 features**: MONTH, DAY_OF_WEEK, DEP_HOUR, SEASON, IS_WEEKEND, DISTANCE + 14 airline dummies.

In [ ]:
import sys
sys.path.insert(0, "..")

import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression, LinearRegression
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMRegressor

from modules.data_loader import load_flights_clean, build_classification_split, build_regression_split
from modules.evaluation import (
    print_classification_report,
    plot_confusion_matrix,
    plot_roc_curves,
    print_regression_report,
    plot_residuals,
    plot_feature_importance,
)

sns.set_style('whitegrid')

In [ ]:
df = load_flights_clean()
print(f"Dataset: {df.shape}")

---
## Parte 1 — Classificação Binária

**Target**: `IS_DELAYED` (DEPARTURE_DELAY > 15 minutos)

**Desafio**: A classe majoritária (pontual) representa ~82% dos dados. Um modelo que prevê sempre "pontual" já atinge 82% de accuracy. Precisamos de métricas além de accuracy (precision, recall, F1, ROC-AUC) e tratamento de imbalance.

In [ ]:
X_train, X_test, y_train, y_test = build_classification_split(df)

print(f"Treino: {X_train.shape}, Teste: {X_test.shape}")
print(f"\nBalance treino:")
print((y_train.value_counts(normalize=True) * 100).round(1))
print(f"\nBalance teste:")
print((y_test.value_counts(normalize=True) * 100).round(1))

### Modelo 1 — Regressão Logística (baseline)

Modelo linear interpretável. Cada coeficiente indica a direção e magnitude do efeito de cada feature na probabilidade de atraso. `solver='saga'` é otimizado para datasets grandes. `class_weight='balanced'` ajusta os pesos inversamente proporcionais à frequência das classes.

In [ ]:
t0 = time.time()
lr = LogisticRegression(solver='saga', max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train, y_train)
lr_time = time.time() - t0
print(f"Tempo de treino: {lr_time:.1f}s")

In [ ]:
y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1]

lr_metrics = print_classification_report(y_test, y_pred_lr, y_prob_lr, "Regressão Logística")
plot_confusion_matrix(y_test, y_pred_lr, "Regressão Logística")

### Modelo 2 — XGBoost

Gradient boosting: combina árvores de decisão fracas iterativamente, onde cada árvore corrige os erros da anterior. Captura não-linearidades e interações entre features.

- `scale_pos_weight=4.6` — razão ~82/18 entre classes, compensa o imbalance
- `early_stopping_rounds=20` — para o treino se a métrica não melhorar em 20 rodadas
- `eval_set` — 10% do treino como validação para early stopping

In [ ]:
# Separar validação para early stopping (10% do treino)
rng = np.random.default_rng(42)
n_train = len(X_train)
val_size = int(n_train * 0.1)
val_idx = rng.choice(n_train, val_size, replace=False)
train_mask = np.ones(n_train, dtype=bool)
train_mask[val_idx] = False

X_tr, X_val = X_train.iloc[train_mask], X_train.iloc[val_idx]
y_tr, y_val = y_train.iloc[train_mask], y_train.iloc[val_idx]

t0 = time.time()
xgb_clf = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=4.6,
    early_stopping_rounds=20,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
)
xgb_clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=50)
xgb_time = time.time() - t0
print(f"\nTempo de treino: {xgb_time:.1f}s")
print(f"Melhor iteração: {xgb_clf.best_iteration}")

In [ ]:
y_pred_xgb = xgb_clf.predict(X_test)
y_prob_xgb = xgb_clf.predict_proba(X_test)[:, 1]

xgb_metrics = print_classification_report(y_test, y_pred_xgb, y_prob_xgb, "XGBoost")
plot_confusion_matrix(y_test, y_pred_xgb, "XGBoost")

In [ ]:
# Comparação lado a lado
comparison_clf = pd.DataFrame([lr_metrics, xgb_metrics]).set_index('model')
print("Comparação — Classificação:")
print(comparison_clf.round(4).to_string())

# ROC curves sobrepostas
plot_roc_curves({
    "Regressão Logística": (y_test, y_prob_lr),
    "XGBoost": (y_test, y_prob_xgb),
})

# Feature importance XGBoost
plot_feature_importance(xgb_clf, list(X_train.columns), "XGBoost Classificação")

### Análise — Classificação

**Comparação**: A tabela acima mostra as métricas de ambos os modelos. O XGBoost tipicamente supera a Regressão Logística em datasets tabulares com interações não-lineares, mas a LR serve como baseline interpretável.

**Pontos-chave**:
- `class_weight='balanced'` / `scale_pos_weight` melhoram recall à custa de precision
- ROC-AUC é a métrica mais informativa quando há imbalance
- As features pré-partida (hora, dia, mês, companhia, distância) capturam padrões gerais mas não eventos específicos — performance limitada é esperada

---
## Parte 2 — Regressão

**Target**: `DEPARTURE_DELAY` (minutos), filtrado para voos com delay > 0.

Este é um problema mais difícil — as features pré-partida têm poder limitado para prever a **magnitude exata** do atraso. Um R² baixo é esperado e é um achado válido.

In [ ]:
X_train_r, X_test_r, y_train_r, y_test_r = build_regression_split(df)

print(f"Treino: {X_train_r.shape}, Teste: {X_test_r.shape}")
print(f"\nEstatísticas do target (DEPARTURE_DELAY > 0):")
print(y_train_r.describe().round(1))

### Modelo 3 — Regressão Linear (baseline)

Baseline interpretável para regressão. Assume relação linear entre features e target.

In [ ]:
t0 = time.time()
lin_reg = LinearRegression()
lin_reg.fit(X_train_r, y_train_r)
linreg_time = time.time() - t0
print(f"Tempo de treino: {linreg_time:.1f}s")

y_pred_linreg = lin_reg.predict(X_test_r)
linreg_metrics = print_regression_report(y_test_r, y_pred_linreg, "Regressão Linear")
plot_residuals(y_test_r, y_pred_linreg, "Regressão Linear")

### Modelo 4 — LightGBM Regressor

Gradient boosting com histogram-based splitting — 2 a 3x mais rápido que XGBoost para datasets grandes. Ideal para os ~2M de registros da regressão.

In [ ]:
# Validação para early stopping
rng = np.random.default_rng(42)
n_train_r = len(X_train_r)
val_size_r = int(n_train_r * 0.1)
val_idx_r = rng.choice(n_train_r, val_size_r, replace=False)
train_mask_r = np.ones(n_train_r, dtype=bool)
train_mask_r[val_idx_r] = False

X_tr_r, X_val_r = X_train_r.iloc[train_mask_r], X_train_r.iloc[val_idx_r]
y_tr_r, y_val_r = y_train_r.iloc[train_mask_r], y_train_r.iloc[val_idx_r]

t0 = time.time()
lgbm_reg = LGBMRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)
lgbm_reg.fit(
    X_tr_r, y_tr_r,
    eval_set=[(X_val_r, y_val_r)],
    callbacks=[
        __import__('lightgbm').early_stopping(20),
        __import__('lightgbm').log_evaluation(50),
    ],
)
lgbm_time = time.time() - t0
print(f"\nTempo de treino: {lgbm_time:.1f}s")
print(f"Melhor iteração: {lgbm_reg.best_iteration_}")

In [ ]:
y_pred_lgbm = lgbm_reg.predict(X_test_r)
lgbm_metrics = print_regression_report(y_test_r, y_pred_lgbm, "LightGBM")
plot_residuals(y_test_r, y_pred_lgbm, "LightGBM")

### Modelo 5 — XGBoost Regressor

Comparação direta com LightGBM: mesmo framework conceitual (gradient boosting), implementação diferente. Permite avaliar o trade-off velocidade vs acurácia.

In [ ]:
t0 = time.time()
xgb_reg = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    early_stopping_rounds=20,
    random_state=42,
    n_jobs=-1,
)
xgb_reg.fit(X_tr_r, y_tr_r, eval_set=[(X_val_r, y_val_r)], verbose=50)
xgb_reg_time = time.time() - t0
print(f"\nTempo de treino: {xgb_reg_time:.1f}s")
print(f"Melhor iteração: {xgb_reg.best_iteration}")

In [ ]:
y_pred_xgb_r = xgb_reg.predict(X_test_r)
xgb_reg_metrics = print_regression_report(y_test_r, y_pred_xgb_r, "XGBoost Regressor")
plot_residuals(y_test_r, y_pred_xgb_r, "XGBoost Regressor")

In [ ]:
# Comparação regressão
comparison_reg = pd.DataFrame([linreg_metrics, lgbm_metrics, xgb_reg_metrics]).set_index('model')
print("Comparação — Regressão:")
print(comparison_reg.round(4).to_string())

# Tempo de treino
print(f"\nTempo de treino:")
print(f"  Regressão Linear: {linreg_time:.1f}s")
print(f"  LightGBM:         {lgbm_time:.1f}s")
print(f"  XGBoost:          {xgb_reg_time:.1f}s")

# Feature importance dos modelos de regressão
plot_feature_importance(lgbm_reg, list(X_train_r.columns), "LightGBM Regressão")
plot_feature_importance(xgb_reg, list(X_train_r.columns), "XGBoost Regressão")

### Análise — Regressão

**R² esperado baixo**: As features pré-partida (hora, dia, mês, companhia, distância) capturam padrões **estatísticos gerais**, mas a magnitude exata de um atraso individual depende de fatores **operacionais em tempo real** (clima no momento, problemas mecânicos, congestionamento de pista) que não estão nas features.

Um R² baixo **não é falha do modelo** — é um achado que revela a **limitação intrínseca** de prever magnitude de atraso com informações pré-partida.

**LightGBM vs XGBoost**: Comparação de velocidade e acurácia. LightGBM tipicamente treina mais rápido (histogram-based splitting) com acurácia similar.

---
## Conclusões

### Classificação (IS_DELAYED)
- 2 modelos comparados: Regressão Logística (baseline linear) vs XGBoost (ensemble não-linear)
- Imbalance tratado com `class_weight='balanced'` / `scale_pos_weight`
- Features mais importantes: tipicamente DEP_HOUR, MONTH, companhias específicas

### Regressão (DEPARTURE_DELAY)
- 3 modelos comparados: Regressão Linear vs LightGBM vs XGBoost
- R² limitado — achado esperado dada a natureza das features pré-partida
- LightGBM vs XGBoost: trade-off velocidade/acurácia documentado

### Limitações
- Dataset de 2015 apenas — generalização temporal não validada
- Features limitadas a informações pré-partida
- Outliers extremos (>300 min) podem afetar RMSE desproporcionalmente
- 20 features de contexto geral — sem dados meteorológicos ou operacionais em tempo real